## Example 1: Basic Import and Inspection\n
\n
Basic import test and inspection of graphql_relay package structure.\n

In [ ]:
# Example 1: Relay Connection Specification
from graphql import GraphQLObjectType, GraphQLField, GraphQLString, GraphQLInt, GraphQLSchema, GraphQLArgument, graphql_sync
from graphql_relay import connection_from_array, connection_definitions, connection_args

print("Relay Connection Specification:")
print("=" * 32)

# Sample data - a list of users
users_data = [
    {"id": "1", "name": "Alice", "email": "alice@example.com", "age": 30},
    {"id": "2", "name": "Bob", "email": "bob@example.com", "age": 25},
    {"id": "3", "name": "Charlie", "email": "charlie@example.com", "age": 35},
    {"id": "4", "name": "Diana", "email": "diana@example.com", "age": 28},
    {"id": "5", "name": "Eve", "email": "eve@example.com", "age": 32},
]

print(f"1. Sample data: {len(users_data)} users")

# Define User type
user_type = GraphQLObjectType(
    "User",
    lambda: {
        "id": GraphQLField(GraphQLString),
        "name": GraphQLField(GraphQLString),
        "email": GraphQLField(GraphQLString),
        "age": GraphQLField(GraphQLInt),
    }
)

# Create connection type for User
user_connection_defs = connection_definitions(user_type)

print(f"2. Connection types created:")
print(f"   Connection Type: {user_connection_defs.connection_type.name}")
print(f"   Edge Type: {user_connection_defs.edge_type.name}")

# Define query with connection
query_type = GraphQLObjectType(
    "Query",
    lambda: {
        "users": GraphQLField(
            user_connection_defs.connection_type,
            args=connection_args,
            resolve=lambda obj, info, **args: connection_from_array(
                users_data, args
            )
        )
    }
)

# Create schema
schema = GraphQLSchema(query=query_type)

# Test pagination queries
print(f"\n3. Testing pagination:")

# Query first 2 users
query1 = """
{
  users(first: 2) {
    edges {
      cursor
      node {
        id
        name
        email
      }
    }
    pageInfo {
      hasNextPage
      hasPreviousPage
      startCursor
      endCursor
    }
  }
}
"""

result = graphql_sync(schema, query1)
print(f"   First 2 users:")
if result.data:
    page_info = result.data['users']['pageInfo']
    print(f"     Edges: {len(result.data['users']['edges'])}")
    print(f"     Has next page: {page_info['hasNextPage']}")
    print(f"     Has previous page: {page_info['hasPreviousPage']}")

# Query last 2 users
query2 = """
{
  users(last: 2) {
    edges {
      node {
        name
      }
    }
    pageInfo {
      hasNextPage
      hasPreviousPage
    }
  }
}
"""

result = graphql_sync(schema, query2)
print(f"\n   Last 2 users:")
if result.data:
    page_info = result.data['users']['pageInfo']
    edges = result.data['users']['edges']
    print(f"     Users: {[e['node']['name'] for e in edges]}")
    print(f"     Has next page: {page_info['hasNextPage']}")

# Query with cursor-based pagination
query3 = """
{
  users(first: 2) {
    edges {
      cursor
      node {
        name
      }
    }
  }
}
"""

result = graphql_sync(schema, query3)
if result.data and result.data['users']['edges']:
    cursor = result.data['users']['edges'][-1]['cursor']
    print(f"\n   Cursor-based pagination:")
    print(f"     Got cursor: {cursor}")
    
    # Use cursor to get next page
    query4 = f"""
    {{
      users(first: 2, after: "{cursor}") {{
        edges {{
          node {{
            name
          }}
        }}
        pageInfo {{
          hasNextPage
        }}
      }}
    }}
    """
    
    result = graphql_sync(schema, query4)
    if result.data:
        edges = result.data['users']['edges']
        print(f"     Next page users: {[e['node']['name'] for e in edges]}")

print("\n✅ Relay Connection Specification completed successfully!")

## Example 2: Core Functionality Test\n
\n
Test core functionality of graphql_relay package.\n

In [ ]:
# Example 2: Node Interface and Global Object Identification
from graphql import GraphQLObjectType, GraphQLField, GraphQLString, GraphQLInt, GraphQLSchema, GraphQLNonNull, GraphQLID, GraphQLArgument, graphql_sync
from graphql_relay import node_definitions, from_global_id, to_global_id

print("Node Interface and Global Object Identification:")
print("=" * 47)

# Sample data for different types
users_db = {
    "1": {"id": "1", "name": "Alice", "type": "User"},
    "2": {"id": "2", "name": "Bob", "type": "User"},
}

posts_db = {
    "1": {"id": "1", "title": "GraphQL Relay Tutorial", "authorId": "1", "type": "Post"},
    "2": {"id": "2", "title": "Advanced Relay Patterns", "authorId": "2", "type": "Post"},
}

print(f"1. Sample databases:")
print(f"   Users: {len(users_db)} entries")
print(f"   Posts: {len(posts_db)} entries")

# Function to get node by global ID
def get_node(global_id, info):
    """Resolve node from global ID"""
    type_name, local_id = from_global_id(global_id)
    
    if type_name == "User":
        return users_db.get(local_id)
    elif type_name == "Post":
        return posts_db.get(local_id)
    return None

# Function to get type name from object
def get_type(obj, info, type_):
    """Get GraphQL type name from object"""
    type_name = obj.get("type")
    if type_name == "User":
        return user_type
    elif type_name == "Post":
        return post_type
    return None

# Create node interface - returns 3 values in current version
node_interface, node_field, node_fields = node_definitions(get_node, get_type)

print(f"\n2. Node interface created:")
print(f"   Interface name: {node_interface.name}")
print(f"   Node field type: {type(node_field).__name__}")

# Define User type with node interface
user_type = GraphQLObjectType(
    "User",
    lambda: {
        "id": GraphQLField(
            GraphQLNonNull(GraphQLID),
            resolve=lambda obj, info: to_global_id("User", obj["id"])
        ),
        "name": GraphQLField(GraphQLString),
    },
    interfaces=[node_interface]
)

# Define Post type with node interface
post_type = GraphQLObjectType(
    "Post",
    lambda: {
        "id": GraphQLField(
            GraphQLNonNull(GraphQLID),
            resolve=lambda obj, info: to_global_id("Post", obj["id"])
        ),
        "title": GraphQLField(GraphQLString),
        "author": GraphQLField(
            user_type,
            resolve=lambda obj, info: users_db.get(obj.get("authorId"))
        ),
    },
    interfaces=[node_interface]
)

# Create schema with node field
query_type = GraphQLObjectType(
    "Query",
    lambda: {
        "node": node_field,
        "user": GraphQLField(
            user_type,
            args={"id": GraphQLArgument(GraphQLNonNull(GraphQLString))},
            resolve=lambda obj, info, id: users_db.get(id)
        )
    }
)

schema = GraphQLSchema(query=query_type, types=[user_type, post_type])

# Test global ID encoding/decoding
print(f"\n3. Testing Global ID:")
global_user_id = to_global_id("User", "1")
global_post_id = to_global_id("Post", "1")

print(f"   User ID '1' -> Global ID: {global_user_id}")
print(f"   Post ID '1' -> Global ID: {global_post_id}")

decoded_user = from_global_id(global_user_id)
decoded_post = from_global_id(global_post_id)

print(f"   Decoded User: {decoded_user}")
print(f"   Decoded Post: {decoded_post}")

# Query by global ID
print(f"\n4. Querying by Global ID:")
query = f"""
{{
  node(id: "{global_user_id}") {{
    id
    ... on User {{
      name
    }}
  }}
}}
"""

result = graphql_sync(schema, query)
if result.data:
    print(f"   Query result: {result.data}")

# Query post with author
query2 = f"""
{{
  node(id: "{global_post_id}") {{
    id
    ... on Post {{
      title
      author {{
        name
      }}
    }}
  }}
}}
"""

result = graphql_sync(schema, query2)
if result.data:
    print(f"   Post with author: {result.data}")

print("\n✅ Node Interface and Global Object Identification completed successfully!")

## Example 3: Error Handling and Edge Cases\n
\n
Test error handling and edge cases for graphql_relay.\n

In [ ]:
# Example 3: Relay Mutations with Client Mutation ID
from graphql import GraphQLObjectType, GraphQLField, GraphQLString, GraphQLInt, GraphQLSchema, GraphQLNonNull, graphql_sync, GraphQLInputField
from graphql_relay import mutation_with_client_mutation_id

print("Relay Mutations with Client Mutation ID:")
print("=" * 39)

# Mutable user database
users_database = {
    "1": {"id": "1", "name": "Alice", "email": "alice@example.com"},
    "2": {"id": "2", "name": "Bob", "email": "bob@example.com"},
}

print(f"1. Initial database: {len(users_database)} users")

# Define User type
user_type = GraphQLObjectType(
    "User",
    lambda: {
        "id": GraphQLField(GraphQLString),
        "name": GraphQLField(GraphQLString),
        "email": GraphQLField(GraphQLString),
    }
)

# Create mutation to add a new user
add_user_mutation = mutation_with_client_mutation_id(
    "AddUser",
    input_fields={
        "name": GraphQLInputField(GraphQLNonNull(GraphQLString)),
        "email": GraphQLInputField(GraphQLNonNull(GraphQLString)),
    },
    output_fields={
        "user": GraphQLField(user_type),
        "userEdge": GraphQLField(
            GraphQLObjectType(
                "UserEdge",
                {
                    "node": GraphQLField(user_type),
                    "cursor": GraphQLField(GraphQLString)
                }
            )
        )
    },
    mutate_and_get_payload=lambda info, **input: add_user(input)
)

def add_user(input):
    """Add new user to database"""
    new_id = str(len(users_database) + 1)
    new_user = {
        "id": new_id,
        "name": input["name"],
        "email": input["email"]
    }
    users_database[new_id] = new_user
    
    return {
        "user": new_user,
        "userEdge": {
            "node": new_user,
            "cursor": f"cursor:{new_id}"
        }
    }

# Create mutation to update user
update_user_mutation = mutation_with_client_mutation_id(
    "UpdateUser",
    input_fields={
        "id": GraphQLInputField(GraphQLNonNull(GraphQLString)),
        "name": GraphQLInputField(GraphQLString),
        "email": GraphQLInputField(GraphQLString),
    },
    output_fields={
        "user": GraphQLField(user_type)
    },
    mutate_and_get_payload=lambda info, **input: update_user(input)
)

def update_user(input):
    """Update existing user"""
    user_id = input["id"]
    if user_id in users_database:
        user = users_database[user_id]
        if "name" in input:
            user["name"] = input["name"]
        if "email" in input:
            user["email"] = input["email"]
        return {"user": user}
    return {"user": None}

# Create mutation to delete user
delete_user_mutation = mutation_with_client_mutation_id(
    "DeleteUser",
    input_fields={
        "id": GraphQLInputField(GraphQLNonNull(GraphQLString)),
    },
    output_fields={
        "deletedId": GraphQLField(GraphQLString),
        "success": GraphQLField(GraphQLString)
    },
    mutate_and_get_payload=lambda info, **input: delete_user(input)
)

def delete_user(input):
    """Delete user from database"""
    user_id = input["id"]
    if user_id in users_database:
        del users_database[user_id]
        return {"deletedId": user_id, "success": "true"}
    return {"deletedId": None, "success": "false"}

# Define mutation type
mutation_type = GraphQLObjectType(
    "Mutation",
    lambda: {
        "addUser": add_user_mutation,
        "updateUser": update_user_mutation,
        "deleteUser": delete_user_mutation,
    }
)

# Define query type
query_type = GraphQLObjectType(
    "Query",
    lambda: {
        "users": GraphQLField(
            GraphQLObjectType(
                "UsersList",
                {"items": GraphQLField(GraphQLObjectType("UserItem", {"id": GraphQLField(GraphQLString), "name": GraphQLField(GraphQLString), "email": GraphQLField(GraphQLString)}))}
            ),
            resolve=lambda obj, info: {"items": list(users_database.values())}
        )
    }
)

# Create schema
schema = GraphQLSchema(query=query_type, mutation=mutation_type)

print(f"\n2. Testing mutations:")

# Test add user mutation
print(f"\n   Adding new user 'Charlie':")
add_mutation = """
mutation {
  addUser(input: {name: "Charlie", email: "charlie@example.com", clientMutationId: "1"}) {
    user {
      id
      name
      email
    }
    userEdge {
      cursor
    }
    clientMutationId
  }
}
"""

result = graphql_sync(schema, add_mutation)
if result.data:
    print(f"     New user: {result.data['addUser']['user']}")
    print(f"     Client mutation ID: {result.data['addUser']['clientMutationId']}")
    print(f"     Database now has: {len(users_database)} users")

# Test update user mutation
print(f"\n   Updating user '1' (Alice):")
update_mutation = """
mutation {
  updateUser(input: {id: "1", name: "Alice Smith", clientMutationId: "2"}) {
    user {
      id
      name
      email
    }
    clientMutationId
  }
}
"""

result = graphql_sync(schema, update_mutation)
if result.data:
    print(f"     Updated user: {result.data['updateUser']['user']}")
    print(f"     Client mutation ID: {result.data['updateUser']['clientMutationId']}")

# Test delete user mutation
print(f"\n   Deleting user '2' (Bob):")
delete_mutation = """
mutation {
  deleteUser(input: {id: "2", clientMutationId: "3"}) {
    deletedId
    success
    clientMutationId
  }
}
"""

result = graphql_sync(schema, delete_mutation)
if result.data:
    print(f"     Deleted ID: {result.data['deleteUser']['deletedId']}")
    print(f"     Success: {result.data['deleteUser']['success']}")
    print(f"     Client mutation ID: {result.data['deleteUser']['clientMutationId']}")
    print(f"     Database now has: {len(users_database)} users")

print(f"\n3. Final database state: {list(users_database.values())}")

print("\n✅ Relay Mutations with Client Mutation ID completed successfully!")

## Example 4: Integration and Usage Patterns\n
\n
Test integration patterns and common usage of graphql_relay.\n

In [ ]:
# Example 4: Advanced Connection Features
from graphql import GraphQLObjectType, GraphQLField, GraphQLString, GraphQLInt, GraphQLSchema, GraphQLList, GraphQLArgument, graphql_sync
from graphql_relay import connection_from_array, connection_definitions, cursor_to_offset, offset_to_cursor, connection_args

print("Advanced Connection Features:")
print("=" * 29)

# Large dataset for demonstrating advanced features
products_data = [
    {"id": str(i), "name": f"Product {i}", "price": 10 + i * 5, "category": "Electronics" if i % 2 == 0 else "Clothing"}
    for i in range(1, 21)
]

print(f"1. Dataset: {len(products_data)} products")

# Define Product type
product_type = GraphQLObjectType(
    "Product",
    lambda: {
        "id": GraphQLField(GraphQLString),
        "name": GraphQLField(GraphQLString),
        "price": GraphQLField(GraphQLInt),
        "category": GraphQLField(GraphQLString),
    }
)

# Create connection with custom edge fields
product_connection = connection_definitions(
    product_type,
    edge_fields=lambda: {
        "rank": GraphQLField(
            GraphQLInt,
            resolve=lambda obj, info: cursor_to_offset(obj.cursor) + 1
        )
    }
)

print(f"\n2. Connection with custom edge fields:")
print(f"   Edge type fields: {list(product_connection.edge_type.fields.keys())}")

# Create query with filtering
def resolve_products(obj, info, category=None, **args):
    """Resolve products with optional filtering"""
    filtered_products = products_data
    
    if category:
        filtered_products = [p for p in products_data if p["category"] == category]
    
    return connection_from_array(filtered_products, args)

query_type = GraphQLObjectType(
    "Query",
    lambda: {
        "products": GraphQLField(
            product_connection.connection_type,
            args={
                **connection_args,
                "category": GraphQLArgument(GraphQLString)
            },
            resolve=resolve_products
        )
    }
)

schema = GraphQLSchema(query=query_type)

# Test with custom edge fields
print(f"\n3. Testing custom edge fields (rank):")
query1 = """
{
  products(first: 3) {
    edges {
      rank
      cursor
      node {
        name
        price
      }
    }
    pageInfo {
      hasNextPage
    }
  }
}
"""

result = graphql_sync(schema, query1)
if result.data:
    edges = result.data['products']['edges']
    print(f"   Products with ranks:")
    for edge in edges:
        print(f"     Rank {edge['rank']}: {edge['node']['name']} - ${edge['node']['price']}")

# Test filtering
print(f"\n4. Testing filtering by category:")
query2 = """
{
  electronics: products(first: 3, category: "Electronics") {
    edges {
      node {
        name
        category
      }
    }
  }
  clothing: products(first: 3, category: "Clothing") {
    edges {
      node {
        name
        category
      }
    }
  }
}
"""

result = graphql_sync(schema, query2)
if result.data:
    print(f"   Electronics: {[e['node']['name'] for e in result.data['electronics']['edges']]}")
    print(f"   Clothing: {[e['node']['name'] for e in result.data['clothing']['edges']]}")

# Test cursor utilities
print(f"\n5. Testing cursor utilities:")
cursor1 = offset_to_cursor(0)
cursor2 = offset_to_cursor(5)
cursor3 = offset_to_cursor(10)

print(f"   Offset 0 -> Cursor: {cursor1}")
print(f"   Offset 5 -> Cursor: {cursor2}")
print(f"   Offset 10 -> Cursor: {cursor3}")

offset1 = cursor_to_offset(cursor1)
offset2 = cursor_to_offset(cursor2)
offset3 = cursor_to_offset(cursor3)

print(f"   Cursor {cursor1} -> Offset: {offset1}")
print(f"   Cursor {cursor2} -> Offset: {offset2}")
print(f"   Cursor {cursor3} -> Offset: {offset3}")

# Test backward pagination
print(f"\n6. Testing backward pagination:")
query3 = """
{
  products(last: 3) {
    edges {
      node {
        name
      }
    }
    pageInfo {
      hasPreviousPage
      hasNextPage
    }
  }
}
"""

result = graphql_sync(schema, query3)
if result.data:
    edges = result.data['products']['edges']
    page_info = result.data['products']['pageInfo']
    print(f"   Last 3 products: {[e['node']['name'] for e in edges]}")
    print(f"   Has previous page: {page_info['hasPreviousPage']}")
    print(f"   Has next page: {page_info['hasNextPage']}")

print("\n✅ Advanced Connection Features completed successfully!")

## Example 5: Comprehensive Feature Test\n
\n
Comprehensive test of graphql_relay features and capabilities.\n

In [ ]:
# Example 5: Complete Relay Application Pattern
from graphql import GraphQLObjectType, GraphQLField, GraphQLString, GraphQLInt, GraphQLSchema, GraphQLNonNull, GraphQLID, GraphQLInputField, graphql_sync
from graphql_relay import (
    connection_from_array, 
    connection_definitions,
    node_definitions,
    from_global_id,
    to_global_id,
    mutation_with_client_mutation_id,
    connection_args
)

print("Complete Relay Application Pattern:")
print("=" * 34)

# Application database
app_database = {
    "users": {
        "1": {"id": "1", "name": "Alice", "email": "alice@example.com"},
        "2": {"id": "2", "name": "Bob", "email": "bob@example.com"},
    },
    "comments": {
        "1": {"id": "1", "text": "Great article!", "userId": "1"},
        "2": {"id": "2", "text": "Thanks for sharing", "userId": "2"},
        "3": {"id": "3", "text": "Very helpful", "userId": "1"},
    }
}

print(f"1. Application data:")
print(f"   Users: {len(app_database['users'])}")
print(f"   Comments: {len(app_database['comments'])}")

# Node interface setup
def get_node_by_global_id(global_id, info):
    """Fetch node by global ID"""
    type_name, local_id = from_global_id(global_id)
    if type_name == "User":
        user = app_database["users"].get(local_id)
        if user:
            user["__typename"] = "User"
        return user
    elif type_name == "Comment":
        comment = app_database["comments"].get(local_id)
        if comment:
            comment["__typename"] = "Comment"
        return comment
    return None

def get_node_type(obj, info, type_):
    """Determine GraphQL type from object"""
    typename = obj.get("__typename")
    if typename == "User":
        return user_type
    elif typename == "Comment":
        return comment_type
    return None

# Create node interface - returns 3 values in current version
node_interface, node_field, node_fields = node_definitions(get_node_by_global_id, get_node_type)

print(f"\n2. Node interface configured")

# Define Comment type first
comment_type = GraphQLObjectType(
    "Comment",
    lambda: {
        "id": GraphQLField(
            GraphQLNonNull(GraphQLID),
            resolve=lambda obj, info: to_global_id("Comment", obj["id"])
        ),
        "text": GraphQLField(GraphQLString),
        "author": GraphQLField(
            user_type,
            resolve=lambda obj, info: app_database["users"].get(obj.get("userId"))
        )
    },
    interfaces=[node_interface]
)

# Create comment connection from the comment_type
comment_connection = connection_definitions(comment_type)

# Define User type
user_type = GraphQLObjectType(
    "User",
    lambda: {
        "id": GraphQLField(
            GraphQLNonNull(GraphQLID),
            resolve=lambda obj, info: to_global_id("User", obj["id"])
        ),
        "name": GraphQLField(GraphQLString),
        "email": GraphQLField(GraphQLString),
        "comments": GraphQLField(
            comment_connection.connection_type,
            args=connection_args,
            resolve=lambda obj, info, **args: connection_from_array(
                [c for c in app_database["comments"].values() if c.get("userId") == obj.get("id")],
                args
            )
        )
    },
    interfaces=[node_interface]
)

# Add mutation to create comment
add_comment_mutation = mutation_with_client_mutation_id(
    "AddComment",
    input_fields={
        "userId": GraphQLInputField(GraphQLNonNull(GraphQLString)),
        "text": GraphQLInputField(GraphQLNonNull(GraphQLString)),
    },
    output_fields={
        "comment": GraphQLField(comment_type),
        "user": GraphQLField(user_type)
    },
    mutate_and_get_payload=lambda info, **input: add_comment_to_db(input)
)

def add_comment_to_db(input):
    """Add comment to database"""
    new_id = str(len(app_database["comments"]) + 1)
    new_comment = {
        "id": new_id,
        "text": input["text"],
        "userId": input["userId"],
        "__typename": "Comment"
    }
    app_database["comments"][new_id] = new_comment
    
    user = app_database["users"].get(input["userId"])
    if user:
        user = user.copy()
        user["__typename"] = "User"
    
    return {
        "comment": new_comment,
        "user": user
    }

# Define schema
query_type = GraphQLObjectType(
    "Query",
    lambda: {
        "node": node_field,
        "viewer": GraphQLField(
            user_type,
            resolve=lambda obj, info: {**app_database["users"]["1"], "__typename": "User"}
        )
    }
)

mutation_type = GraphQLObjectType(
    "Mutation",
    lambda: {
        "addComment": add_comment_mutation
    }
)

schema = GraphQLSchema(query=query_type, mutation=mutation_type, types=[user_type, comment_type])

print(f"\n3. Schema created with queries and mutations")

# Test complete application flow
print(f"\n4. Testing complete Relay flow:")

# Query viewer with comments
print(f"\n   a) Query viewer (Alice) with comments:")
query1 = """
{
  viewer {
    id
    name
    comments(first: 2) {
      edges {
        node {
          id
          text
        }
      }
      pageInfo {
        hasNextPage
      }
    }
  }
}
"""

result = graphql_sync(schema, query1)
if result.data and result.data.get('viewer'):
    viewer = result.data['viewer']
    print(f"      Viewer: {viewer['name']}")
    print(f"      Comments: {len(viewer['comments']['edges'])}")
    for edge in viewer['comments']['edges']:
        print(f"        - {edge['node']['text']}")
elif result.errors:
    print(f"      Errors: {result.errors}")

# Add new comment via mutation
print(f"\n   b) Add new comment:")
mutation = """
mutation {
  addComment(input: {userId: "2", text: "Excellent post!", clientMutationId: "mutation1"}) {
    comment {
      id
      text
    }
    user {
      name
      comments(first: 10) {
        edges {
          node {
            text
          }
        }
      }
    }
    clientMutationId
  }
}
"""

result = graphql_sync(schema, mutation)
if result.data and result.data.get('addComment'):
    add_result = result.data['addComment']
    print(f"      New comment: {add_result['comment']['text']}")
    print(f"      User: {add_result['user']['name']}")
    print(f"      User's all comments: {len(add_result['user']['comments']['edges'])}")
    print(f"      Client mutation ID: {add_result['clientMutationId']}")
elif result.errors:
    print(f"      Errors: {result.errors}")

# Query by global node ID
print(f"\n   c) Query comment by global ID:")
comment_global_id = to_global_id("Comment", "1")
query2 = f"""
{{
  node(id: "{comment_global_id}") {{
    id
    ... on Comment {{
      text
      author {{
        name
      }}
    }}
  }}
}}
"""

result = graphql_sync(schema, query2)
if result.data and result.data.get('node'):
    node = result.data['node']
    print(f"      Comment: {node['text']}")
    print(f"      Author: {node['author']['name']}")
elif result.errors:
    print(f"      Errors: {result.errors}")
else:
    print(f"      Node not found")

print(f"\n5. Final database state:")
print(f"   Total comments: {len(app_database['comments'])}")
print(f"   Comments by user:")
for user_id, user in app_database['users'].items():
    user_comments = [c for c in app_database['comments'].values() if c.get('userId') == user_id]
    print(f"     {user['name']}: {len(user_comments)} comments")

print("\n✅ Complete Relay Application Pattern completed successfully!")